<div align="center">

<img width="80%" src="https://user-images.githubusercontent.com/73097560/115834477-dbab4500-a447-11eb-908a-139a6edaec5c.gif"/>


<img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg"
     alt="Python"
     width="80"
     height="80"/>
<img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/docker/docker-original.svg"
     alt="Docker"
     width="80"
     height="80"/>
<img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/scikitlearn/scikitlearn-original.svg"
     alt="scikit-learn"
     width="80"
     height="80"/>

![Python](https://img.shields.io/badge/Python-3.8+-3776AB?style=flat-square&logo=python&logoColor=white)
![Docker](https://img.shields.io/badge/Docker-2496ED?style=flat-square&logo=docker&logoColor=white)
![scikit-learn](https://img.shields.io/badge/scikit--learn-F7931E?style=flat-square&logo=scikit-learn&logoColor=white)

<h1>Docker: ML para Produção — Análise Exploratória</h1>

<img width="80%" src="https://user-images.githubusercontent.com/73097560/115834477-dbab4500-a447-11eb-908a-139a6edaec5c.gif"/>

</div>

### PhD. Julles Mitoura

In [ ]:
# Obs: se você não estiver utilizando o ambiente virtual do módulo, instale as dependências:
# %pip install -q -r ../requirements.txt

---

<div align="center">

## <span style="color:#1E90FF;">Contexto</span>

</div>

Este notebook realiza a **análise exploratória completa** do dataset do trocador de calor (`heat_exchanger.db`), base do case utilizado ao longo do curso.

O objetivo é entender a estrutura, qualidade e comportamento dos dados **antes** de construir qualquer modelo preditivo. As observações aqui levantadas justificam as escolhas de pré-processamento e modelagem feitas nos scripts de produção (`train.py` e `inference.py`).

As etapas desta análise são:

- **Carregamento**: leitura do banco SQLite com todas as colunas disponíveis;
- **Qualidade dos dados**: tipos, valores nulos, intervalos esperados;
- **Estatísticas descritivas**: resumo numérico de cada variável;
- **Evolução temporal**: comportamento das variáveis ao longo do período;
- **Distribuições**: forma e simetria de cada feature;
- **Correlações**: relações lineares entre variáveis;
- **Tendência de degradação**: quantificação e visualização do decaimento da eficiência.

In [ ]:
import os
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

DB_PATH = os.getenv('DB_PATH', '../data/heat_exchanger.db')

---

<div align="center">

## <span style="color:#1E90FF;">Carregamento dos Dados</span>

</div>

O dataset está armazenado num banco **SQLite** com uma única tabela (`heat_exchanger`). Carregamos todas as colunas disponíveis para a análise exploratória — diferente do `train.py`, que seleciona apenas as colunas necessárias para o modelo.

In [ ]:
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query(
    'SELECT * FROM heat_exchanger ORDER BY timestamp',
    conn,
)
conn.close()

df['timestamp'] = pd.to_datetime(df['timestamp'])
df['day_index'] = (df['timestamp'] - df['timestamp'].min()).dt.days

print(f'Shape             : {df.shape}')
print(f'Período           : {df["timestamp"].min().date()} → {df["timestamp"].max().date()}')
print(f'Total de registros: {len(df)}')
print(f'Colunas           : {list(df.columns)}')

df.head()

---

<div align="center">

## <span style="color:#1E90FF;">Qualidade dos Dados</span>

</div>

Verificamos os tipos de cada coluna, a presença de valores nulos e os intervalos observados. Esta etapa é essencial antes de qualquer modelagem — dados com missing values ou outliers extremos exigem tratamento explícito.

In [ ]:
print('=== Tipos de dados ===')
print(df.dtypes)

print('\n=== Valores nulos ===')
nulls = df.isnull().sum()
print(nulls)
print(f'\nTotal de nulos: {nulls.sum()} — dataset {"limpo" if nulls.sum() == 0 else "com missing values"}')

print('\n=== Intervalos das variáveis numéricas ===')
num_cols = df.select_dtypes(include='number').columns.tolist()
for col in num_cols:
    print(f'  {col:<35s}: [{df[col].min():.3f}, {df[col].max():.3f}]')

---

<div align="center">

## <span style="color:#1E90FF;">Estatísticas Descritivas</span>

</div>

O `describe()` fornece um resumo rápido de cada variável numérica: média, desvio padrão, quartis e extremos. Valores de desvio padrão baixos em variáveis de temperatura podem indicar condições operacionais estáveis — o que é esperado para um processo industrial controlado.

In [ ]:
df.drop(columns=['day_index']).describe().round(4)

---

<div align="center">

## <span style="color:#1E90FF;">Evolução Temporal</span>

</div>

Visualizamos o comportamento de todas as variáveis ao longo do tempo. O padrão esperado para um trocador de calor em degradação é:

- **Eficiência térmica**: queda monotônica ao longo do período;
- **Temperaturas**: relativamente estáveis, com flutuações operacionais.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

# --- Eficiência térmica ---
axes[0].plot(df['timestamp'], df['heat_efficiency'], color='steelblue', linewidth=1.5)
axes[0].set_title('Eficiência Térmica ao Longo do Tempo', fontweight='bold')
axes[0].set_ylabel('Eficiência (%)')
axes[0].fill_between(df['timestamp'], df['heat_efficiency'], alpha=0.15, color='steelblue')

# --- Temperaturas de entrada ---
axes[1].plot(df['timestamp'], df['water_inlet_temperature'],
             label='Água (entrada)', color='tomato', linewidth=1.2)
axes[1].plot(df['timestamp'], df['glycol_inlet_temperature'],
             label='Glicol (entrada)', color='darkorange', linewidth=1.2)
axes[1].set_title('Temperaturas de Entrada', fontweight='bold')
axes[1].set_ylabel('Temperatura (°C)')
axes[1].legend()

# --- Temperaturas de saída ---
axes[2].plot(df['timestamp'], df['out_water_temperature'],
             label='Água (saída)', color='seagreen', linewidth=1.2)
axes[2].plot(df['timestamp'], df['out_glycol_temperature'],
             label='Glicol (saída)', color='mediumorchid', linewidth=1.2)
axes[2].set_title('Temperaturas de Saída', fontweight='bold')
axes[2].set_ylabel('Temperatura (°C)')
axes[2].set_xlabel('Data')
axes[2].legend()

plt.tight_layout()
plt.show()

---

<div align="center">

## <span style="color:#1E90FF;">Distribuições</span>

</div>

Histogramas e boxplots revelam a forma das distribuições de cada variável. Distribuições simétricas e sem outliers extremos indicam um processo estável. Assimetrias ou caudas longas podem indicar eventos operacionais anômalos.

In [ ]:
feat_cols = [
    'water_inlet_temperature', 'glycol_inlet_temperature',
    'out_water_temperature', 'out_glycol_temperature', 'heat_efficiency',
]
labels = [
    'Água\n(entrada)', 'Glicol\n(entrada)',
    'Água\n(saída)', 'Glicol\n(saída)', 'Eficiência\ntérmica',
]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))

for i, (col, lbl) in enumerate(zip(feat_cols, labels)):
    # Histograma
    axes[0, i].hist(df[col], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
    axes[0, i].set_title(lbl, fontsize=10)
    axes[0, i].set_ylabel('Frequência' if i == 0 else '')

    # Boxplot
    bp = axes[1, i].boxplot(
        df[col], vert=True, patch_artist=True,
        boxprops=dict(facecolor='steelblue', alpha=0.6),
        medianprops=dict(color='tomato', linewidth=2),
    )
    axes[1, i].set_ylabel('Valor' if i == 0 else '')

plt.suptitle('Distribuições — Histogramas e Boxplots', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---

<div align="center">

## <span style="color:#1E90FF;">Correlações</span>

</div>

A matriz de correlação quantifica a relação linear entre pares de variáveis. Para este dataset, esperamos:

- **`day_index` × `heat_efficiency`**: correlação negativa forte — a eficiência cai com o tempo;
- **Temperaturas entre si**: possivelmente correlacionadas por física do processo;
- **Temperaturas × `heat_efficiency`**: correlação que pode revelar quais variáveis físicas mais impactam a eficiência.

In [ ]:
corr_cols = [
    'water_inlet_temperature', 'glycol_inlet_temperature',
    'out_water_temperature', 'out_glycol_temperature',
    'heat_efficiency', 'day_index',
]
corr = df[corr_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.3f',
    cmap='coolwarm', vmin=-1, vmax=1, center=0,
    square=True, linewidths=0.5,
)
plt.title('Matriz de Correlação (triângulo inferior)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

<div align="center">

## <span style="color:#1E90FF;">Tendência de Degradação</span>

</div>

Esta é a análise central do EDA. Quantificamos a taxa de degradação da eficiência térmica ao longo do tempo ajustando uma regressão linear simples sobre `day_index`.

Se a tendência for **monotônica e linear**, um modelo de regressão linear é a escolha mais adequada para capturar e **extrapolar** essa degradação — modelos baseados em árvores não extrapolam além do range observado.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X = df[['day_index']].values
y = df['heat_efficiency'].values

model = LinearRegression().fit(X, y)
trend = model.predict(X)
residuals = y - trend

print('=== Regressão Linear: Eficiência × Tempo ===')
print(f'  Coef. angular : {model.coef_[0]:.6f}% por dia')
print(f'  Intercepto   : {model.intercept_:.4f}%')
print(f'  R²            : {r2_score(y, trend):.4f}')
print(f'  Variação total: {trend[-1] - trend[0]:.4f}% em {int(df["day_index"].max())} dias')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Série + tendência ---
axes[0].scatter(df['timestamp'], y, s=15, alpha=0.6, color='steelblue', label='Medição real', zorder=2)
axes[0].plot(
    df['timestamp'], trend, color='tomato', linewidth=2,
    label=f'Tendência linear ({model.coef_[0]:.4f}%/dia)',
)
axes[0].set_title('Eficiência com Tendência Linear Ajustada', fontweight='bold')
axes[0].set_xlabel('Data')
axes[0].set_ylabel('Eficiência (%)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=20)

# --- Resíduos ---
axes[1].scatter(df['timestamp'], residuals, s=15, alpha=0.6, color='steelblue')
axes[1].axhline(0, color='tomato', linewidth=1.5, linestyle='--')
axes[1].set_title(f'Resíduos da Tendência Linear (std={residuals.std():.4f}%)', fontweight='bold')
axes[1].set_xlabel('Data')
axes[1].set_ylabel('Resíduo (%)')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

print('\nConclusão: a tendência é linear, os resíduos são aleatórios (sem padrão estrutural).')
print('→ Regressão Linear é a escolha correta para modelar e extrapolar esta degradação.')